# 02 (full data) · e5 embeddings and candidate buckets for every train and test record

The same method as `02_e5_embeddings.ipynb`, run on **all** records instead of a sample:
* text `name_norm | addr_norm` from 01's `processed/` files, encoded with
  `intfloat/multilingual-e5-small` (MIT), in fp16 on both GPUs. Names and addresses written in an
  Indian script are transliterated to Latin letters from the original text instead (see below).
* per country, every S1 gets its `TOP_K` most similar S2/S3 records (its **bucket**), found by
  exact search on mean-centered embeddings, on both GPUs
* **extra candidates** (`EXTRA_SOURCES`) are added to each bucket from three other searches, which
  miss different matches than e5 does (see "Extra candidate sources")

Both splits are processed:
* **train** buckets are what the full LightGBM notebook learns from
* **test** buckets are what it predicts on, and they become `candidate_pairs.tsv`

Only the buckets are saved. The embeddings of all 24M records would not fit in Kaggle's 20 GB
output limit.

**Input:**
* 01's `processed/` folder, from a run with `FULL_RUN=true`. 01 now also writes `test_s2_*` and
  `test_s3_*`, which the test buckets need.
* `train_ground_truth.tsv`, used only for the bucket-recall report.

**Output** (`embeddings_full/multilingual-e5-small/`):

| File | Content |
|---|---|
| `buckets/<split>_<country>.parquet` | one row per (S1, candidate), sorted by S1 and then score: `s1_entity_id`, `rank` (position in the bucket), `candidate_entity_id`, `candidate_source`, `score` (centered cosine), `cosine` (original e5 cosine), `country`, and which search found it: `e5_rank`, `rev_rank`, `name_rank` (empty = not found by that search), `addr_key` (1 = same house number and street word) |
| `bucket_recall_train.csv` | per country and candidate set (e5 top-20/30/40/50, e5 top-30 plus each extra source, all): pairs per S1, share of true matches inside, and the best score a perfect matcher could reach |
| `manifest.json` | settings, record counts and timings |

**Run on Kaggle:**
1. Accelerator: GPU T4 x2. Internet on, to download the model (or attach it and set `E5_MODEL`).
2. Attach the competition data and the output of 01 (`FULL_RUN=true`), then Run all.
3. About 24M texts are encoded. My estimate is roughly an hour on two T4s, plus a few minutes of
   search per country and roughly 30-45 minutes for the extra sources (mostly the name search). A
   bucket file that already exists is skipped, so a re-run in the same session continues where it
   stopped.

Settings are in the first code cell and can also be set as environment variables: `PROCESSED_DIR`,
`DATA_ROOT`, `EMB_FULL_DIR`, `SPLITS`, `TOP_K`, `E5_MODEL`, `TEXT_MODE`, `COUNTRIES`, `EXTRA_SOURCES`,
`REVERSE_TOP`, `ADDRESS_KEY_CAP`, `NAME_TOP`. `EXTRA_SOURCES=""` gives the e5-only buckets of version 2.

In [ ]:
import csv
import gc
import glob
import inspect
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

SPLITS = [s.strip() for s in os.environ.get("SPLITS", "train,test").split(",") if s.strip()]
TOP_K = int(os.environ.get("TOP_K", 30))                       # candidates kept per S1 (the matcher uses all 30)
TEXT_MODE = os.environ.get("TEXT_MODE", "fixed")               # fixed | processed | latin | raw (see "Build the buckets")
COUNTRIES = [c for c in os.environ.get("COUNTRIES", "").split(",") if c]   # empty = every country (testing aid)
MODEL_NAME = os.environ.get("E5_MODEL", "intfloat/multilingual-e5-small")
PREFIX = "query: "           # E5 expects "query: " for symmetric similarity
MAX_SEQ_LENGTH = 128         # tokens; names + addresses are short
ENCODE_CHUNK = 1_000_000     # texts per encoding call; keeps memory flat for countries with millions of records
# extra candidates added to the e5 top-k (see "Extra candidate sources"); "" = e5 only
EXTRA_SOURCES = [x for x in os.environ.get("EXTRA_SOURCES", "reverse,address,name").split(",") if x]
REVERSE_K = 5                # reverse search: best S1 kept per S2/S3 record (their rank is a feature for 03)
REVERSE_TOP = int(os.environ.get("REVERSE_TOP", 2))            # ... and pairs added when the S1 is among its best 2
ADDRESS_KEY_CAP = int(os.environ.get("ADDRESS_KEY_CAP", 20))   # skip house number + street keys shared by more candidates
NAME_TOP = int(os.environ.get("NAME_TOP", 5))                  # name 3-gram matches added per S1
NAME_MAX_DF = 0.02           # 3-grams in more than 2% of the names are ignored (10x faster; most of the gain kept)
REPORT_K = (20, 30, 40, 50)  # e5 top-k sizes compared with the extra sources in the train recall report
assert TEXT_MODE in {"fixed", "processed", "latin", "raw"}, TEXT_MODE
assert set(SPLITS) <= {"train", "test"}, SPLITS
assert set(EXTRA_SOURCES) <= {"reverse", "address", "name"}, EXTRA_SOURCES


def find_processed_dir():
    """Folder with 01's per-country parquet: $PROCESSED_DIR, ./processed, /kaggle/working, any Kaggle input."""
    candidates = [Path(os.environ["PROCESSED_DIR"])] if os.environ.get("PROCESSED_DIR") else []
    candidates += [Path.cwd() / "processed", Path("/kaggle/working/processed")]
    candidates += sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c.is_dir() and any(c.glob("train_s1_*.parquet")):
            return c
    raise FileNotFoundError("processed/ not found: run 01_eda_preprocessing.ipynb with FULL_RUN=true or set PROCESSED_DIR")


def find_ground_truth():
    """train_ground_truth.tsv under $DATA_ROOT, /kaggle/input or the local dataset folder; None if absent."""
    roots = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", str(Path.cwd().parent / "Data" / "student_resource" / "dataset")]
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for h in glob.glob(os.path.join(root, "**", "train_ground_truth.tsv"), recursive=True)
                    if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


PROCESSED_DIR = find_processed_dir()
GT_FILE = find_ground_truth()
OUT_DIR = Path(os.environ.get("EMB_FULL_DIR") or ("/kaggle/working/embeddings_full" if os.path.isdir("/kaggle/working")
                                                  else "embeddings_full")) / MODEL_NAME.rstrip("/").split("/")[-1]
BUCKET_DIR = OUT_DIR / "buckets"
BUCKET_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else []   # all used (T4 x2 on Kaggle)
if os.environ.get("GPU_LIST"):                                  # testing aid, e.g. "cuda:0,cuda:0"
    GPUS = os.environ["GPU_LIST"].split(",")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# fp16 speeds up tensor-core GPUs (T4, A100) but is ~4x slower on GTX 16xx cards (measured on a GTX 1650).
USE_FP16 = DEVICE == "cuda" and "GTX 16" not in GPU_NAME
BATCH_SIZE = 1024 if USE_FP16 else 512                         # texts per GPU batch when encoding
CHUNK_SIZE = 16 * BATCH_SIZE                                   # texts per work unit handed to one GPU process
SEARCH_DTYPE = torch.float16 if USE_FP16 else torch.float32    # first scoring pass (fp16 = tensor cores)
RESCORE_EXTRA = 32                                             # the top k + 32 of that pass are re-scored in fp32
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory if DEVICE == "cuda" else 0
print(f"processed: {PROCESSED_DIR}\nground truth: {GT_FILE}\noutput: {OUT_DIR}")
print(f"device: {len(GPUS)} x {GPU_NAME} | fp16: {USE_FP16} | batch: {BATCH_SIZE} | splits: {SPLITS} | top-k: {TOP_K} | "
      f"text mode: {TEXT_MODE} | extra sources: {EXTRA_SOURCES or 'none'}")

## Inventory of `processed/`
Every (split, country) with S1 records gets buckets when S2/S3 records exist for it. If
`test_s2_*` / `test_s3_*` are missing, 01 ran with the old file list: re-run it with `FULL_RUN=true`.

In [ ]:
FILE_RE = re.compile(r"^(train|test)_(s[123])_(.+)\.parquet$")
INVENTORY = pd.DataFrame([{"split": m.group(1), "source": m.group(2), "country": m.group(3), "path": f,
                           "rows": pq.ParquetFile(f).metadata.num_rows}
                          for f in sorted(PROCESSED_DIR.glob("*.parquet")) if (m := FILE_RE.match(f.name))])
assert len(INVENTORY), f"no <split>_<source>_<country>.parquet files in {PROCESSED_DIR}"
if (INVENTORY["rows"] <= 40_000).all():
    print("WARNING: every file has <= 40,000 rows: processed/ looks like 01's sample mode. Run 01 with FULL_RUN=true.")
for split in SPLITS:
    if not ((INVENTORY["split"] == split) & INVENTORY["source"].isin(["s2", "s3"])).any():
        print(f"WARNING: no {split}_s2_* / {split}_s3_* files, so no {split} buckets can be built. "
              "Re-run 01 (FULL_RUN=true) with the updated file list.")
INVENTORY.pivot_table(index=["split", "country"], columns="source", values="rows", aggfunc="sum").fillna(0).astype(int)

## Encoding on the GPUs
* Each unique text is encoded once, in fp16 on T4s, `BATCH_SIZE` texts per batch.
* Texts are sorted by length, so each batch holds texts of similar length (little padding).
* With several GPUs, one encoding process runs per GPU, and work units of `CHUNK_SIZE` texts go
  to whichever GPU is free.
* Texts are encoded `ENCODE_CHUNK` at a time and stored as fp16 straight away, which keeps memory
  flat even for countries with millions of records.

In [ ]:
from sentence_transformers import SentenceTransformer

_model, _pool = None, None


def get_model():
    """Load the e5 model once (kept on the CPU when it is copied to several GPUs)."""
    global _model
    if _model is None:
        _model = SentenceTransformer(MODEL_NAME, device="cpu" if len(GPUS) > 1 else DEVICE)
        _model.max_seq_length = MAX_SEQ_LENGTH
        if USE_FP16:
            _model.half()
    return _model


def get_pool():
    """One encoding process per GPU, started once and reused for every file."""
    global _pool
    if _pool is None:
        _pool = get_model().start_multi_process_pool(target_devices=GPUS)
    return _pool


def stop_pool():
    """Stop the encoding processes."""
    global _pool
    if _pool is not None:
        SentenceTransformer.stop_multi_process_pool(_pool)
        _pool = None


def encode(batch):
    """float32 embeddings of a list of texts, on all GPUs."""
    model = get_model()
    if len(GPUS) > 1:
        if "pool" in inspect.signature(model.encode).parameters:          # sentence-transformers >= 5
            return model.encode(batch, pool=get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE,
                                convert_to_numpy=True, show_progress_bar=False)
        return model.encode_multi_process(batch, get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE)
    return model.encode(batch, batch_size=BATCH_SIZE, convert_to_numpy=True, show_progress_bar=False)


def embed_all(texts, label):
    """Unit-norm float16 embeddings aligned with texts (each unique text encoded once, longest first)."""
    codes, uniq = pd.factorize(np.asarray(texts, dtype=object))
    order = np.argsort(-np.fromiter((len(t) for t in uniq), np.int64, len(uniq)), kind="stable")
    out, t0 = None, time.time()
    for s in range(0, len(order), ENCODE_CHUNK):
        idx = order[s:s + ENCODE_CHUNK]
        vecs = np.asarray(encode([PREFIX + t for t in uniq[idx]]), dtype=np.float32)
        vecs /= np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
        if out is None:
            out = np.empty((len(uniq), vecs.shape[1]), np.float16)
        out[idx] = vecs.astype(np.float16)
        done = min(s + ENCODE_CHUNK, len(order))
        print(f"  {label}: {done:,}/{len(uniq):,} unique texts, {done / max(time.time() - t0, 1e-9):,.0f} texts/s")
    return out[codes]

## Exact search on the GPUs
* Each GPU centers the S2/S3 embeddings (subtracts their mean and re-normalizes, as in notebook
  02), keeps a copy, and takes its share of the S1 query blocks.
* A block is scored against all S2/S3 vectors in one matrix product (fp16 on T4s). Its best
  k + 32 per S1 are re-scored in fp32, and the top k are kept.
* The original e5 cosine of every bucket pair is then computed on the GPU as well.
* Without a GPU, the same search runs with NumPy on the CPU (slow; meant for small tests).

In [ ]:
def _normalize(x):
    return x / x.norm(dim=1, keepdim=True).clamp_min(1e-12)


def exact_search(base16, queries16, mu, k):
    """Top-k rows of base16 for every row of queries16, by inner product of mean-centered, re-normalized vectors."""
    n, d = base16.shape
    scores = np.empty((len(queries16), k), np.float32)
    hits = np.empty((len(queries16), k), np.int64)
    if not GPUS:
        base = base16.astype(np.float32) - mu
        base /= np.maximum(np.linalg.norm(base, axis=1, keepdims=True), 1e-12)
        for s in range(0, len(queries16), 1024):
            q = queries16[s:s + 1024].astype(np.float32) - mu
            q /= np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
            sims = q @ base.T
            top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            order = np.argsort(-np.take_along_axis(sims, top, axis=1), axis=1)
            hits[s:s + 1024] = np.take_along_axis(top, order, axis=1)
            scores[s:s + 1024] = np.take_along_axis(sims, hits[s:s + 1024], axis=1)
        return scores, hits
    shortlist = min(n, k + RESCORE_EXTRA)
    elem = 2 if SEARCH_DTYPE == torch.float16 else 4
    rows = int(np.clip((GPU_MEMORY // 8) // (n * elem), 64, 4096))    # query rows per block: score block <= 1/8 of GPU memory
    blocks = [(s, min(s + rows, len(queries16))) for s in range(0, len(queries16), rows)]

    def run(device, my_blocks):
        with torch.cuda.device(device):
            mu_t = torch.from_numpy(mu).to(device)
            base = torch.empty((n, d), dtype=SEARCH_DTYPE, device=device)
            for s in range(0, n, 1_000_000):                           # center in chunks (no fp32 copy of everything)
                chunk = torch.from_numpy(base16[s:s + 1_000_000]).to(device).float() - mu_t
                base[s:s + len(chunk)] = _normalize(chunk).to(SEARCH_DTYPE)
            del chunk
            for s, e in my_blocks:
                q = _normalize(torch.from_numpy(queries16[s:e]).to(device).float() - mu_t)
                cand = torch.topk(q.to(SEARCH_DTYPE) @ base.T, shortlist, dim=1).indices
                rescored = torch.einsum("qd,qkd->qk", q, base[cand].float())
                top = torch.topk(rescored, k, dim=1)
                scores[s:e] = top.values.cpu().numpy()
                hits[s:e] = torch.gather(cand, 1, top.indices).cpu().numpy()
            del base
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(len(GPUS)) as ex:
        list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
    return scores, hits


def pair_score(a, a_rows, b, b_rows, mu, chunk=500_000):
    """Search score (dot product of mean-centered, re-normalized vectors) of a[a_rows[j]] and b[b_rows[j]] for every j."""
    out = np.empty(len(a_rows), np.float32)
    if GPUS:
        with torch.cuda.device(GPUS[0]):
            ta, tb = torch.from_numpy(a).to(GPUS[0]), torch.from_numpy(b).to(GPUS[0])
            mu_t = torch.from_numpy(mu).to(GPUS[0])
            for s in range(0, len(a_rows), chunk):
                ia = torch.from_numpy(a_rows[s:s + chunk]).to(GPUS[0])
                ib = torch.from_numpy(b_rows[s:s + chunk]).to(GPUS[0])
                out[s:s + chunk] = (_normalize(ta[ia].float() - mu_t) * _normalize(tb[ib].float() - mu_t)).sum(dim=1).cpu().numpy()
            del ta, tb
            torch.cuda.empty_cache()
        return out
    for s in range(0, len(a_rows), chunk):
        x = a[a_rows[s:s + chunk]].astype(np.float32) - mu
        y = b[b_rows[s:s + chunk]].astype(np.float32) - mu
        x /= np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
        y /= np.maximum(np.linalg.norm(y, axis=1, keepdims=True), 1e-12)
        out[s:s + chunk] = np.einsum("ij,ij->i", x, y)
    return out


def pair_cosine(a, a_rows, b, b_rows, chunk=500_000):
    """Dot product of a[a_rows[j]] and b[b_rows[j]] for every j (original e5 cosine), in chunks, on a GPU when there is one."""
    out = np.empty(len(a_rows), np.float32)
    if GPUS:
        with torch.cuda.device(GPUS[0]):
            ta, tb = torch.from_numpy(a).to(GPUS[0]), torch.from_numpy(b).to(GPUS[0])
            for s in range(0, len(a_rows), chunk):
                ia = torch.from_numpy(a_rows[s:s + chunk]).to(GPUS[0])
                ib = torch.from_numpy(b_rows[s:s + chunk]).to(GPUS[0])
                out[s:s + chunk] = (ta[ia].float() * tb[ib].float()).sum(dim=1).cpu().numpy()
            del ta, tb
            torch.cuda.empty_cache()
        return out
    for s in range(0, len(a_rows), chunk):
        out[s:s + chunk] = np.einsum("ij,ij->i", a[a_rows[s:s + chunk]].astype(np.float32),
                                     b[b_rows[s:s + chunk]].astype(np.float32))
    return out

## Ground truth (train buckets report only)
The labels are used only to report how many true matches the train buckets contain. They do not
influence the embeddings or the search.

In [ ]:
OWNER, N_TRUE_ALL = None, None
if "train" in SPLITS and GT_FILE is not None:
    gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
    N_TRUE_ALL = gt.set_index("source1_entity_id")["matched_entity_ids"].map(lambda s: len(s.split(",")) if s else 0)
    pairs = gt[gt["matched_entity_ids"] != ""]
    pairs = pairs.assign(entity_id=pairs["matched_entity_ids"].str.split(",")).explode("entity_id")
    OWNER = pd.Series(pairs["source1_entity_id"].to_numpy(), index=pairs["entity_id"].to_numpy())
    del gt, pairs
    print(f"{len(OWNER):,} train S2/S3 records have a true S1; {int((N_TRUE_ALL == 0).sum()):,} train S1 are singletons")
else:
    print("no ground truth (or train not in SPLITS): the bucket-recall report is skipped")


def owner_rows(s1_ids, c_ids):
    """Row in s1_ids of each candidate's true S1 (-1 when it has none or it is not in this country)."""
    pos = pd.Series(np.arange(len(s1_ids)), index=s1_ids)
    return pos.reindex(OWNER.reindex(c_ids).to_numpy()).fillna(-1).to_numpy(np.int64)


def recall_row(country, label, keys, n_c, owner, n_true):
    """Train only: pairs per S1, share of true matches among the candidate pairs (keys = s1 row * n_c + candidate row)
    and the best macro F0.5 a perfect matcher could reach on them."""
    keys = np.unique(keys)
    q, c = keys // n_c, keys % n_c
    found = np.bincount(q[owner[c] == q], minlength=len(n_true))
    with np.errstate(divide="ignore", invalid="ignore"):
        r = found / n_true
        f = np.where(n_true == 0, 1.0, np.where(found > 0, 1.25 * r / (0.25 + r), 0.0))
    return {"country": country, "candidates": label, "S1": len(n_true), "pairs per S1": round(len(keys) / len(n_true), 1),
            "true matches in bucket": round(float(found.sum() / max(n_true.sum(), 1)), 4),
            "ceiling macro F0.5": round(float(f.mean()), 4)}

## Extra candidate sources
e5 compares name and address together, so it misses some matches. On a dense regional test bed
(same density as the full data), each of these found 4-10x more missed true matches per extra
pair than simply taking a bigger e5 top-k (India top-30 recall 91.4% -> 93.3% with address + name,
for 3.7 extra pairs per S1; the reverse search needs the embeddings, so it was not measured there). The train recall report below measures them at full scale.

| Source | Pair added when | Catches |
|---|---|---|
| `reverse` | the S1 is among the `REVERSE_TOP` best S1 of the S2/S3 record (the same e5 score, searched from the candidate's side) | matches pushed below rank 30 by lookalikes in crowded cities |
| `address` | same house number and first street word (keys shared by more than `ADDRESS_KEY_CAP` candidates are skipped) | same address, different or garbled name |
| `name` | among the S1's `NAME_TOP` most similar names by character 3-grams (TF-IDF cosine; 3-grams in more than 2% of names ignored) | same name, different or empty address; typos |

Each bucket row records which searches found it (`e5_rank`, `rev_rank`, `addr_key`, `name_rank`),
and 03 uses these as features. The address text is the original address (Indian scripts
transliterated with `anyascii`), and the name is 01's `core_name` (the transliterated original for
Indian scripts). The street parsing is the same as in 03.

The name search multiplies sparse 3-gram matrices on the GPUs (a block of S1 names against all
candidate names). A block that does not fit in GPU memory is split in two.

In [ ]:
import unicodedata

_ACCENTS = {}
for _cp in range(0xC0, 0x250):
    _base = unicodedata.normalize("NFKD", chr(_cp))[0]
    if _base.isascii() and _base != chr(_cp):
        _ACCENTS[_cp] = _base
FOLD = str.maketrans(_ACCENTS)
RE_WORD = re.compile(r"\w+")
STREET_TYPES = frozenset("rue avenue av ave boulevard bd blvd boul chemin che place pl route rte allee impasse imp quai "
                         "cours crs square faubourg fbg road rd street st lane ln drive dr way court ct circle cir "
                         "highway hwy parkway pkwy terrace marg".split())
STREET_STOP = STREET_TYPES | frozenset("r ch all sq bis ter de du des la le les l d of the and et en sur suite ste unit "
                                       "apt apartment building bldg floor fl po box door no nr n number h hno flat shop "
                                       "plot near opp".split())


def fold_text(s):
    """Lower case with Latin accents removed (other scripts unchanged)."""
    return s.lower().translate(FOLD)


def words(s):
    """Folded word tokens; numbers without leading zeros ('0058' -> '58')."""
    return [(w.lstrip("0") or "0") if w.isdigit() else w for w in RE_WORD.findall(fold_text(s))]


def street_of(address):
    """(street name, house number): the first comma part with a number or a street word, minus numbers, street
    words and stop words. '15 bis Rue de Barbieux, Roubaix' -> ('barbieux', '15')."""
    house = ""
    for part in address.split(","):
        w = words(part)
        nums = [x for x in w if x.isdigit()]
        if nums and not house:
            house = nums[0]
        if nums or any(x in STREET_TYPES for x in w):
            name = " ".join(x for x in w if not x.isdigit() and x not in STREET_STOP and len(x) > 1)
            if name:
                return name, house
    return "", house


def read_extra(paths):
    """entity_ids, names and addresses for the extra sources, in the order of read_texts."""
    df = pd.concat([pd.read_parquet(p, columns=["entity_id", "business_name", "core_name", "business_address"]) for p in paths],
                   ignore_index=True)
    names = [(anyascii(r) if RE_NATIVE.search(r) else c).lower() for r, c in
             zip(df["business_name"].fillna("").astype(str), df["core_name"].fillna("").astype(str))]
    addrs = [anyascii(a) if RE_NATIVE.search(a) else a for a in df["business_address"].fillna("").astype(str)]
    return df["entity_id"].astype(str).to_numpy(dtype=object), np.asarray(names, dtype=object), addrs


def address_pairs(s1_addr, c_addr):
    """(S1 row, candidate row) with the same house number and first street word."""
    def keys(addrs):
        codes, uniq = pd.factorize(np.asarray(addrs, dtype=object))
        k = []
        for a in uniq:
            street, house = street_of(a)
            k.append(f"{house}|{street.split()[0]}" if house and street else "")
        return np.asarray(k, dtype=object)[codes]
    left = pd.DataFrame({"q": np.arange(len(s1_addr)), "key": keys(s1_addr)})
    right = pd.DataFrame({"c": np.arange(len(c_addr)), "key": keys(c_addr)})
    left, right = left[left["key"] != ""], right[right["key"] != ""]
    right = right[right["key"].map(right["key"].value_counts()) <= ADDRESS_KEY_CAP]
    j = left.merge(right, on="key")
    return j["q"].to_numpy(np.int64), j["c"].to_numpy(np.int64)


def _csr_to(m, device):
    return torch.sparse_csr_tensor(torch.from_numpy(m.indptr.astype(np.int64)), torch.from_numpy(m.indices.astype(np.int64)),
                                   torch.from_numpy(m.data), size=m.shape, device=device)


def _top_per_row(r, c, v, t):
    """Keep the t best (c, v) per row r; returns (row, col, rank)."""
    order = torch.argsort(-v, stable=True)
    order = order[torch.argsort(r[order], stable=True)]              # grouped by row, best first
    r, c = r[order], c[order]
    pos = torch.arange(len(r), device=r.device) - torch.searchsorted(r, r)
    keep = pos < t
    return r[keep].cpu().numpy(), c[keep].cpu().numpy(), pos[keep].cpu().numpy()


def name_pairs(s1_names, c_names, rows=256):
    """(S1 row, candidate row, rank) of the NAME_TOP candidates per S1 with the most similar name
    (cosine of character 3-gram TF-IDF vectors), with sparse matrix products on the GPUs."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2, max_df=NAME_MAX_DF, sublinear_tf=True,
                          dtype=np.float32)
    try:
        vec.fit(np.concatenate([s1_names, c_names]))
    except ValueError:                                               # no 3-gram left (tiny inputs)
        return (np.empty(0, np.int64),) * 3
    a, bt = vec.transform(s1_names).tocsr(), vec.transform(c_names).T.tocsr()     # S1 x V, V x candidates
    blocks = [(s, min(s + rows, a.shape[0])) for s in range(0, a.shape[0], rows)]
    if not GPUS:                                                     # CPU (small tests only)
        out = []
        for s, e in blocks:
            p = (a[s:e] @ bt).tocoo()
            r, c, k = _top_per_row(torch.from_numpy(p.row.astype(np.int64)), torch.from_numpy(p.col.astype(np.int64)),
                                   torch.from_numpy(p.data), NAME_TOP)
            out.append((r + s, c, k))
        return tuple(np.concatenate(x) for x in zip(*out)) if out else (np.empty(0, np.int64),) * 3

    def run(device, my_blocks):
        out = []
        with torch.cuda.device(device):
            bt_g = _csr_to(bt, device)
            todo = list(my_blocks)
            while todo:
                s, e = todo.pop()
                try:
                    p = (_csr_to(a[s:e], device) @ bt_g).to_sparse_coo().coalesce()
                    r, c = p.indices()
                    r, c, k = _top_per_row(r, c, p.values(), NAME_TOP)
                    out.append((r + s, c, k))
                    del p
                except (torch.cuda.OutOfMemoryError, RuntimeError) as err:   # too big for the GPU: split the block
                    if "memory" not in str(err).lower() and "cusparse" not in str(err).lower():
                        raise
                    torch.cuda.empty_cache()
                    assert e - s > 1, "a single name block does not fit on the GPU"
                    m = (s + e) // 2
                    todo += [(s, m), (m, e)]
            del bt_g
            torch.cuda.empty_cache()
        return out

    with ThreadPoolExecutor(len(GPUS)) as ex:
        parts = [x for out in ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]) for x in out]
    return tuple(np.concatenate(x) for x in zip(*parts)) if parts else (np.empty(0, np.int64),) * 3


def lookup(all_keys, keys, values, fill=np.nan):
    """values of keys placed at their position in all_keys (sorted, unique); fill elsewhere."""
    out = np.full(len(all_keys), fill, np.float32)
    if len(keys):
        pos = np.minimum(np.searchsorted(all_keys, keys), len(all_keys) - 1)
        ok = all_keys[pos] == keys
        out[pos[ok]] = np.asarray(values, np.float32)[ok] if np.ndim(values) else values
    return out

## Build the buckets
One (split, country) at a time: read the texts, encode S1 and S2+S3, search, save, free memory.

**Text given to e5** (`TEXT_MODE`, default `fixed`):
* Latin-script records: `name_norm | addr_norm` from 01, unchanged.
* Names or addresses with a native script (Hindi, Telugu, Malayalam, Tamil, ...): transliterated to
  Latin letters from the original text with `anyascii` (ISC license), e.g. "లోటస్ కేర్" -> "lots ker".
  01's cleaning strips combining marks, which deletes the vowel signs of Indian scripts, so these
  names reached e5 garbled. On a dense regional test bed, 57% of the true matches with an
  Indian-script name were missing from the top-20 buckets (4.7% for the other matches).

In [ ]:
TEXT_COLS = {"fixed": ["name_norm", "addr_norm", "business_name", "business_address"],
             "processed": ["name_norm", "addr_norm"], "latin": ["name_lat", "legal_form", "addr_norm"],
             "raw": ["business_name", "business_address"]}[TEXT_MODE]
if TEXT_MODE == "fixed" or EXTRA_SOURCES:
    import importlib.util
    import subprocess
    import sys
    if importlib.util.find_spec("anyascii") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anyascii"])
    from anyascii import anyascii
# a text has a native script when it has a character outside Latin, Latin-1/Extended and general punctuation
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")


def latin(raw, cleaned):
    """cleaned (01's text) for Latin-script text, else the transliterated original."""
    return " ".join(anyascii(raw).lower().split()) if RE_NATIVE.search(raw) else cleaned


def read_texts(paths):
    """entity_ids and e5 texts (TEXT_MODE, as in notebook 02) of the records in the given processed files."""
    df = pd.concat([pd.read_parquet(p, columns=["entity_id"] + TEXT_COLS) for p in paths], ignore_index=True)
    for c in TEXT_COLS:
        df[c] = df[c].fillna("").astype(str)
    if TEXT_MODE == "raw":
        names, addrs = df["business_name"], df["business_address"]
    elif TEXT_MODE == "latin":
        names, addrs = (df["name_lat"] + " " + df["legal_form"]).str.strip(), df["addr_norm"]
    elif TEXT_MODE == "fixed":
        names = [latin(r, c) for r, c in zip(df["business_name"], df["name_norm"])]
        addrs = [latin(r, c) for r, c in zip(df["business_address"], df["addr_norm"])]
    else:
        names, addrs = df["name_norm"], df["addr_norm"]
    texts = [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]
    return df["entity_id"].astype(str).to_numpy(dtype=object), texts


def files(split, sources, country):
    sel = INVENTORY[(INVENTORY["split"] == split) & INVENTORY["source"].isin(sources) & (INVENTORY["country"] == country)]
    return sel.sort_values("source")["path"].tolist()


manifest_path = OUT_DIR / "manifest.json"
MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
SETTINGS = {"model": MODEL_NAME, "prefix": PREFIX, "max_seq_length": MAX_SEQ_LENGTH, "text_mode": TEXT_MODE,
            "top_k": TOP_K, "processed_dir": str(PROCESSED_DIR), "extra_sources": EXTRA_SOURCES, "reverse_top": REVERSE_TOP,
            "address_key_cap": ADDRESS_KEY_CAP, "name_top": NAME_TOP, "name_max_df": NAME_MAX_DF}
if MANIFEST.get("settings") != SETTINGS:
    MANIFEST = {"settings": SETTINGS, "done": {}, "recall": []}
T_ALL = time.time()
for split in SPLITS:
    for country in sorted(INVENTORY.loc[(INVENTORY["split"] == split) & (INVENTORY["source"] == "s1"), "country"].unique()):
        if COUNTRIES and country not in COUNTRIES:
            continue
        key, dst = f"{split}_{country}", BUCKET_DIR / f"{split}_{country}.parquet"
        if key in MANIFEST["done"] and dst.exists():
            print(f"{key}: already built, skipped")
            continue
        cand_paths = files(split, ["s2", "s3"], country)
        if not cand_paths:
            print(f"{key}: no S2/S3 records, no buckets (its S1 get empty candidate lists)")
            continue
        t0 = time.time()
        s1_ids, s1_texts = read_texts(files(split, ["s1"], country))
        c_ids, c_texts = read_texts(cand_paths)
        if not len(c_ids) or not len(s1_ids):
            print(f"{key}: no records to search, skipped")
            continue
        print(f"{key}: {len(s1_ids):,} S1 and {len(c_ids):,} S2/S3 records")
        s1_emb = embed_all(s1_texts, f"{key} S1")
        c_emb = embed_all(c_texts, f"{key} S2/S3")
        del s1_texts, c_texts
        t_embed = time.time() - t0

        t1 = time.time()
        n_s1, n_c = len(s1_ids), len(c_ids)
        mu = c_emb.mean(axis=0, dtype=np.float64).astype(np.float32)[None, :]
        report = split == "train" and OWNER is not None
        k = min(TOP_K, n_c)
        k_search = min(max((k,) + REPORT_K) if report else k, n_c)
        _, hits = exact_search(c_emb, s1_emb, mu, k_search)
        e5_keys = np.repeat(np.arange(n_s1, dtype=np.int64), k) * n_c + hits[:, :k].ravel()
        e5_rank = np.tile(np.arange(k), n_s1)
        t_search = time.time() - t1

        t2 = time.time()
        extra = {}                                   # source -> (pair keys, value stored in the bucket)
        if "reverse" in EXTRA_SOURCES:
            kr = min(REVERSE_K, n_s1)
            _, rhits = exact_search(s1_emb, c_emb, mu, kr)             # the best S1 of every S2/S3 record
            extra["reverse"] = (rhits.ravel() * n_c + np.repeat(np.arange(n_c, dtype=np.int64), kr), np.tile(np.arange(kr), n_c))
            del rhits
        if "address" in EXTRA_SOURCES or "name" in EXTRA_SOURCES:
            x_s1 = read_extra(files(split, ["s1"], country))
            x_c = read_extra(cand_paths)
            assert (x_s1[0] == s1_ids).all() and (x_c[0] == c_ids).all(), "record order differs between readers"
            if "address" in EXTRA_SOURCES:
                q, c = address_pairs(x_s1[2], x_c[2])
                extra["address"] = (q * n_c + c, np.ones(len(q)))
            if "name" in EXTRA_SOURCES:
                q, c, r = name_pairs(x_s1[1], x_c[1])
                extra["name"] = (q.astype(np.int64) * n_c + c, r)
            del x_s1, x_c
        added = {"reverse": lambda kv: kv[0][kv[1] < REVERSE_TOP]}                 # reverse pairs added to the bucket
        add_keys = {src: added.get(src, lambda kv: kv[0])(kv) for src, kv in extra.items()}
        all_keys = np.unique(np.concatenate([e5_keys] + list(add_keys.values())))
        qi, ci = all_keys // n_c, all_keys % n_c
        score = pair_score(s1_emb, qi, c_emb, ci, mu)
        order = np.lexsort((-score, qi))                             # grouped by S1, best score first
        all_keys, qi, ci, score = all_keys[order], qi[order], ci[order], score[order]
        starts = np.r_[0, np.flatnonzero(qi[1:] != qi[:-1]) + 1]
        rank = np.arange(len(qi)) - np.repeat(starts, np.diff(np.r_[starts, len(qi)]))
        sorted_keys = np.sort(all_keys)
        back = np.argsort(all_keys)                                  # position in all_keys of each sorted key

        def attach(src_keys, values, fill=np.nan):
            out = np.empty(len(all_keys), np.float32)
            out[back] = lookup(sorted_keys, src_keys, values, fill)
            return out
        frame = pd.DataFrame({"s1_entity_id": s1_ids[qi], "rank": rank.astype(np.int16), "candidate_entity_id": c_ids[ci]})
        frame["candidate_source"] = frame["candidate_entity_id"].str[:2]
        frame["score"] = score
        frame["cosine"] = pair_cosine(s1_emb, qi, c_emb, ci)
        frame["country"] = country
        frame["e5_rank"] = attach(e5_keys, e5_rank)
        frame["rev_rank"] = attach(*extra["reverse"]) if "reverse" in extra else np.nan
        frame["addr_key"] = attach(extra["address"][0], 1.0, 0.0).astype(np.int8) if "address" in extra else 0
        frame["name_rank"] = attach(*extra["name"]) if "name" in extra else np.nan
        t_extra = time.time() - t2
        frame.to_parquet(dst, index=False)
        new_pairs = {src: round(float(np.isin(v, e5_keys, invert=True).sum()) / n_s1, 2) for src, v in add_keys.items()}
        print(f"{key}: {len(frame) / n_s1:.1f} pairs per S1 (e5 top-{k} + extra, new pairs per S1 by source: {new_pairs})")

        if report:
            owner, n_true = owner_rows(s1_ids, c_ids), N_TRUE_ALL.reindex(s1_ids).fillna(0).to_numpy()
            q_all = np.repeat(np.arange(n_s1, dtype=np.int64), k_search) * n_c
            for kk in REPORT_K:
                if kk <= k_search:
                    MANIFEST["recall"].append(recall_row(country, f"e5 top-{kk}", (q_all + hits.ravel()).reshape(n_s1, k_search)[:, :kk].ravel(),
                                                         n_c, owner, n_true))
            for src, v in add_keys.items():
                MANIFEST["recall"].append(recall_row(country, f"e5 top-{k} + {src}", np.r_[e5_keys, v], n_c, owner, n_true))
            if add_keys:
                MANIFEST["recall"].append(recall_row(country, f"e5 top-{k} + all extra (saved)", all_keys, n_c, owner, n_true))
            del q_all
        MANIFEST["done"][key] = {"s1": len(s1_ids), "s2_s3": len(c_ids), "pairs": len(frame), "new_pairs_per_s1": new_pairs,
                                 "embed_s": round(t_embed), "search_s": round(t_search), "extra_s": round(t_extra)}
        manifest_path.write_text(json.dumps(MANIFEST, indent=1), encoding="utf-8")
        print(f"{key}: saved {len(frame):,} pairs | encode {t_embed / 60:.1f} min, search {t_search / 60:.1f} min, "
              f"extra sources {t_extra / 60:.1f} min")
        del s1_ids, c_ids, s1_emb, c_emb, hits, qi, ci, frame, all_keys, e5_keys, extra, add_keys
        gc.collect()
        if GPUS:
            torch.cuda.empty_cache()
stop_pool()
print(f"all buckets done in {(time.time() - T_ALL) / 60:.1f} min")
pd.DataFrame([{"bucket": k, **v} for k, v in MANIFEST["done"].items()])

## How many true matches do the train buckets hold?
For train S1 (all of them, at full scale): the share of their true matches that are inside the
bucket, and the macro F0.5 a perfect matcher would reach using only those buckets. The same
numbers are expected for test, which is built the same way.

Rows: the e5 top-k alone for several k, the e5 top-30 plus each extra source, and everything
(what is saved). Compare an extra source with a bigger e5 top-k at a similar "pairs per S1": the
source is worth its cost when it adds more true matches for the same number of extra pairs.

In [ ]:
RECALL = pd.DataFrame(MANIFEST["recall"])
if len(RECALL):
    RECALL.to_csv(OUT_DIR / "bucket_recall_train.csv", index=False)
    order = list(dict.fromkeys(RECALL["candidates"]))
    display(RECALL.pivot(index="candidates", columns="country",
                         values=["pairs per S1", "true matches in bucket", "ceiling macro F0.5"]).reindex(order))
print("bucket files:", sorted(p.name for p in BUCKET_DIR.glob("*.parquet")))